In [1]:
import os
import sys

PROJECT_ROOT = r"C:\sourcecode\Nhom_10_Big_Data"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Current working directory:", os.getcwd())
print("Project root added:", PROJECT_ROOT in sys.path)

Current working directory: C:\sourcecode\Nhom_10_Big_Data\src\ml
Project root added: True


In [2]:
from src.config.spark_session import get_spark
from src.config.hdfs_config import SUPERSTORE_DATASET
from src.config.schema import SUPERSTORE_SCHEMA

print("HDFS path:", SUPERSTORE_DATASET)

HDFS path: hdfs://master:9000/bigdata/superstore/input/G10_dataset.csv


In [3]:
spark = get_spark("G10_MLlib_Loss_Classification")

spark

In [4]:
df = (
    spark.read
    .option("header", True)
    .schema(SUPERSTORE_SCHEMA)
    .csv(SUPERSTORE_DATASET)
)

print("Số dòng:", df.count())
print("Số cột:", len(df.columns))

df.printSchema()

Số dòng: 110000
Số cột: 24
root
 |-- Category: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Market: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- State: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- weeknum: integer (nullable = true)

In [5]:
from pyspark.sql.functions import when, col

df_cls = df.withColumn(
    "label",
    when(col("Profit") < 0, 1.0).otherwise(0.0)
)

df_cls.select("Profit", "label").show(10)

+-------+-----+
| Profit|label|
+-------+-----+
| 9.3312|  0.0|
| 9.2928|  0.0|
| 9.8418|  0.0|
|53.2608|  0.0|
| 3.1104|  0.0|
| 6.5856|  0.0|
| 9.3312|  0.0|
| 5.8604|  0.0|
| 24.219|  0.0|
|23.0864|  0.0|
+-------+-----+
only showing top 10 rows


In [6]:
from pyspark.sql.functions import round as spark_round

total_count = df_cls.count()

label_distribution = (
    df_cls
    .groupBy("label")
    .count()
    .withColumn("ratio_pct", spark_round(col("count") * 100 / total_count, 2))
    .orderBy("label")
)

label_distribution.show()

+-----+-----+---------+
|label|count|ratio_pct|
+-----+-----+---------+
|  0.0|83347|    75.77|
|  1.0|26653|    24.23|
+-----+-----+---------+



In [7]:
numeric_features = [
    "Sales",
    "Quantity",
    "Discount",
    "Shipping_Cost"
]

categorical_features = [
    "Category",
    "Sub_Category",
    "Segment",
    "Market",
    "Region",
    "Ship_Mode",
    "Order_Priority"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['Sales', 'Quantity', 'Discount', 'Shipping_Cost']
Categorical features: ['Category', 'Sub_Category', 'Segment', 'Market', 'Region', 'Ship_Mode', 'Order_Priority']


In [8]:
from pyspark.sql.functions import col, isnan, when, sum as spark_sum

check_columns = numeric_features + ["Profit"]

invalid_exprs = []

for c in check_columns:
    invalid_exprs.append(
        spark_sum(
            when(
                col(c).isNull() |
                isnan(col(c)) |
                (col(c) == float("inf")) |
                (col(c) == float("-inf")),
                1
            ).otherwise(0)
        ).alias(c)
    )

df_cls.select(invalid_exprs).show()

+-----+--------+--------+-------------+------+
|Sales|Quantity|Discount|Shipping_Cost|Profit|
+-----+--------+--------+-------------+------+
|  598|     643|       0|          643|   643|
+-----+--------+--------+-------------+------+



In [9]:
valid_condition = None

for c in numeric_features + ["Profit"]:
    condition = (
        col(c).isNotNull() &
        (~isnan(col(c))) &
        (col(c) != float("inf")) &
        (col(c) != float("-inf"))
    )

    if valid_condition is None:
        valid_condition = condition
    else:
        valid_condition = valid_condition & condition

before_count = df_cls.count()

df_cls = df_cls.filter(valid_condition)

after_count = df_cls.count()

print("Số dòng trước khi loại dòng lỗi:", before_count)
print("Số dòng sau khi loại dòng lỗi:", after_count)
print("Số dòng bị loại:", before_count - after_count)

Số dòng trước khi loại dòng lỗi: 110000
Số dòng sau khi loại dòng lỗi: 109357
Số dòng bị loại: 643


In [10]:
invalid_exprs = []

for c in numeric_features + ["Profit"]:
    invalid_exprs.append(
        spark_sum(
            when(
                col(c).isNull() |
                isnan(col(c)) |
                (col(c) == float("inf")) |
                (col(c) == float("-inf")),
                1
            ).otherwise(0)
        ).alias(c)
    )

df_cls.select(invalid_exprs).show()

+-----+--------+--------+-------------+------+
|Sales|Quantity|Discount|Shipping_Cost|Profit|
+-----+--------+--------+-------------+------+
|    0|       0|       0|            0|     0|
+-----+--------+--------+-------------+------+



In [11]:
from pyspark.sql.functions import lit

total_count = df_cls.count()

count_0 = df_cls.filter(col("label") == 0).count()
count_1 = df_cls.filter(col("label") == 1).count()

weight_0 = total_count / (2 * count_0)
weight_1 = total_count / (2 * count_1)

print("count_0:", count_0)
print("count_1:", count_1)
print("weight_0:", weight_0)
print("weight_1:", weight_1)

df_cls = df_cls.withColumn(
    "class_weight",
    when(col("label") == 0, lit(weight_0)).otherwise(lit(weight_1))
)

df_cls.select("label", "class_weight").show(10)

count_0: 82704
count_1: 26653
weight_0: 0.661134890694525
weight_1: 2.051495141259896
+-----+-----------------+
|label|     class_weight|
+-----+-----------------+
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
|  0.0|0.661134890694525|
+-----+-----------------+
only showing top 10 rows


In [12]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in categorical_features
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_ohe"
    )
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

print("Assembler inputs:")
for item in assembler_inputs:
    print("-", item)

Assembler inputs:
- Sales
- Quantity
- Discount
- Shipping_Cost
- Category_ohe
- Sub_Category_ohe
- Segment_ohe
- Market_ohe
- Region_ohe
- Ship_Mode_ohe
- Order_Priority_ohe


In [13]:
train_df, test_df = df_cls.randomSplit([0.8, 0.2], seed=10)

print("Train:", train_df.count())
print("Test:", test_df.count())

print("Phân phối nhãn trong train:")
train_df.groupBy("label").count().orderBy("label").show()

print("Phân phối nhãn trong test:")
test_df.groupBy("label").count().orderBy("label").show()

Train: 87607
Test: 21750
Phân phối nhãn trong train:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|66145|
|  1.0|21462|
+-----+-----+

Phân phối nhãn trong test:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|16559|
|  1.0| 5191|
+-----+-----+



In [14]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    seed=10
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

model = pipeline.fit(train_df)

predictions = model.transform(test_df)

predictions.select(
    "Sales",
    "Quantity",
    "Discount",
    "Shipping_Cost",
    "Profit",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+--------+--------+-------------+------+-----+----------+----------------------------------------+
|Sales|Quantity|Discount|Shipping_Cost|Profit|label|prediction|probability                             |
+-----+--------+--------+-------------+------+-----+----------+----------------------------------------+
|128.0|6       |0.0     |19.48        |0.0   |0.0  |0.0       |[0.816382911511913,0.18361708848808692] |
|102.0|2       |0.0     |8.42         |46.8  |0.0  |0.0       |[0.8275032395971158,0.17249676040288422]|
|170.0|3       |0.0     |22.56        |25.56 |0.0  |0.0       |[0.835280658435887,0.16471934156411303] |
|26.0 |2       |0.0     |5.31         |1.02  |0.0  |0.0       |[0.813197915808594,0.18680208419140593] |
|68.0 |6       |0.0     |5.93         |10.8  |0.0  |0.0       |[0.8281349384038396,0.17186506159616047]|
|10.0 |1       |0.0     |0.88         |0.66  |0.0  |0.0       |[0.7741403780222516,0.22585962197774848]|
|196.0|6       |0.0     |9.99         |90.0  |0.0  |0.0

In [15]:
confusion_matrix = (
    predictions
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_matrix.show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|15270|
|  0.0|       1.0| 1289|
|  1.0|       0.0|  522|
|  1.0|       1.0| 4669|
+-----+----------+-----+



In [16]:
tp = predictions.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = predictions.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = predictions.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = predictions.filter((col("label") == 1) & (col("prediction") == 0)).count()

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

TP: 4669
TN: 15270
FP: 1289
FN: 522


In [18]:
import pandas as pd

total = tp + tn + fp + fn

accuracy = (tp + tn) / total if total > 0 else 0

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
)

false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0

metrics_data = [
    ("Accuracy", accuracy),
    ("Precision", precision),
    ("Recall / Sensitivity", recall),
    ("Specificity", specificity),
    ("F1-score", f1),
    ("False Positive Rate", false_positive_rate),
    ("False Negative Rate", false_negative_rate)
]

metrics_pd = pd.DataFrame(metrics_data, columns=["Metric", "Value"])
metrics_pd["Value"] = metrics_pd["Value"].round(4)

metrics_pd

,Metric,Value
0,Accuracy,0.9167
1,Precision,0.7837
2,Recall / Sensitivity,0.8994
3,Specificity,0.9222
4,F1-score,0.8376
5,False Positive Rate,0.0778
6,False Negative Rate,0.1006
